# Data Preparation: Analysis Variables and Data Transformation

## Overview
This notebook prepares the filtered and validated survey data (`survey_clean.tsv`) enriched by engineering derived variables and transformations needed for the analysis presented in the paper. It uses custom transformation functions from the `src/data_transformations.py`.

## Purpose
- Load survey data and filter to valid respondents (Q43 chatbot question answered)
- Engineer derived variables: demographics, chatbot usage classification, experience indices, literacy scores, intent frequencies
- Add Likert scale questions as categorical variables
- Export enriched dataset to `survey_clean_var.tsv` for downstream analyses (robustness checks, visualization, etc.)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_transformations import (
    add_demographic_variables,
    classify_chatbot_users,
    calculate_lt_experience_index,
    calculate_literacy_index,
    map_intent_frequencies,
    add_likert_variables
)

print("Imports successful")

## Load Raw Data

In [ ]:
# Define paths
data_dir = Path.cwd().parent
raw_data_file = data_dir / "survey_clean.tsv"
output_file = data_dir / "survey_clean_var.tsv"

# Load raw survey data (skip translated header row)
df = pd.read_csv(raw_data_file, sep="\t", skiprows=[1], encoding='utf-8')

print(f"\nLoaded {len(df)} rows, {len(df.columns)} columns")
print(f"Column names: {list(df.columns)[:121]}...")

## Apply Transformations

In [ ]:
# Step 1: Add demographic variables
print("Step 1: Adding demographic variables...")
df = add_demographic_variables(df)
print(f"  ✓ Added: EducationGroup, GeographyGroup, GenderGroup, IncomeGroup, AgeGroup, Education_STEM, Eduarea")

In [ ]:
# Step 2: Classify chatbot users
print("Step 2: Classifying chatbot users...")
df = classify_chatbot_users(df)
print(f"  ✓ Added: chatbot_user")
print(f"    - Users (Yes): {(df['chatbot_user'] == 'Yes').sum()}")
print(f"    - Non-users (No): {(df['chatbot_user'] == 'No').sum()}")

In [ ]:
# Step 3: Calculate LT experience index
print("Step 3: Calculating LT experience index...")
df = calculate_lt_experience_index(df)
print(f"  ✓ Added: LT_exp (range 0-6)")
print(f"    - Mean: {df['LT_exp'].mean():.2f}")
print(f"    - Std: {df['LT_exp'].std():.2f}")
print(f"    - Distribution:\n{df['LT_exp'].value_counts().sort_index()}")

In [ ]:
# Step 4: Calculate literacy index
print("Step 4: Calculating literacy index...")
df = calculate_literacy_index(df)
print(f"  ✓ Added: lt_lit01 (0-4), lt_lit (0-1)")
print(f"    - Mean lt_lit: {df['lt_lit'].mean():.3f}")
print(f"    - Std lt_lit: {df['lt_lit'].std():.3f}")

In [ ]:
# Step 5: Map intent frequencies
print("Step 5: Mapping intent frequencies...")
df = map_intent_frequencies(df)
print(f"  ✓ Added: InfoRetrieval_freq, ProblemSolving_freq, Learning_freq,")
print(f"           ContentCreation_freq, Entertainment_freq, Creativity_freq")
print(f"    - Values: 0=Never, 1=<Monthly, 2=Monthly, 3=Weekly")

In [ ]:
# Step 6: Recode Likert variables
print("Step 6: Recoding Likert scales (centered at 0)...")
df = add_likert_variables(df)
print(f"  ✓ Added: knowledge, prepared, limitations, potential,")
print(f"           recognize_errors, distinguishai, Education, Bias_awareness")
print(f"    - Range: -2 (strongly disagree) to +2 (strongly agree)")
print(f"    - Zero point: 'Non so' (neutral/don't know)")

## Verify Data

In [ ]:
# Check for missing values in key derived variables
print("Missing values in derived variables:")
derived_cols = [
    'EducationGroup', 'GeographyGroup', 'GenderGroup', 'IncomeGroup', 'AgeGroup',
    'Education_STEM', 'chatbot_user', 'LT_exp', 'lt_lit01', 'lt_lit',
    'InfoRetrieval_freq', 'ProblemSolving_freq', 'Learning_freq',
    'ContentCreation_freq', 'Entertainment_freq', 'Creativity_freq',
    'knowledge', 'prepared', 'limitations', 'potential',
    'recognize_errors', 'distinguishai', 'Education', 'Bias_awareness'
]

missing_summary = df[derived_cols].isnull().sum()
print(missing_summary[missing_summary > 0])

if (missing_summary > 0).sum() == 0:
    print("✓ No missing values in derived variables")

In [ ]:
# Show final dataset structure
print(f"\nFinal dataset: {len(df)} rows × {len(df.columns)} columns")
print(f"\nDerived variables added:")
for col in derived_cols:
    dtype = df[col].dtype
    print(f"  - {col}: {dtype}")

## Save Output

In [ ]:
# Save enriched dataset
df.to_csv(output_file, sep="\t", index=False, encoding='utf-8')

print(f"  Rows: {len(df)}")
print(f"  Columns: {len(df.columns)}")

In [ ]:
# Verify saved file
df_verify = pd.read_csv(output_file, sep="\t", encoding='utf-8')
print(f"\n✓ Verification: Loaded {len(df_verify)} rows × {len(df_verify.columns)} columns")

# Check that all derived variables are present
missing_cols = [col for col in derived_cols if col not in df_verify.columns]
if missing_cols:
    print(f"  ⚠ WARNING: Missing columns: {missing_cols}")
else:
    print(f"  ✓ All derived variables present in saved file")